In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from preprocessing import train_dataset, dev_dataset, test_dataset, label_columns # Đảm bảo file này có trong cùng thư mục
import json
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import numpy as np
from tqdm import tqdm # Đảm bảo đã cài đặt: pip install tqdm

print(f"Số lượng mẫu train: {len(train_dataset)}")
print(f"Số lượng mẫu dev: {len(dev_dataset)}")
print(f"Số lượng mẫu test: {len(test_dataset)}")
print(f"Tổng số nhãn (classes): {len(label_columns)}")
print(f"Danh sách nhãn: {label_columns}")

Các cột trong file Excel: ['file_name', 'Location', 'Food', 'Room', 'Facilities', 'Service', 'Public_area']

📊 Số lượng mẫu theo từng nhãn:
Location: 0 mẫu
Food: 321 mẫu
Room: 2357 mẫu
Facilities: 612 mẫu
Service: 57 mẫu
Public_area: 3096 mẫu
Số lượng mẫu train: 5428
Số lượng mẫu dev: 1789
Số lượng mẫu test: 1841
Tổng số nhãn (classes): 6
Danh sách nhãn: ['Location', 'Food', 'Room', 'Facilities', 'Service', 'Public_area']


In [3]:
# ----- Cấu hình -----
BATCH_SIZE = 32
EPOCHS = 5 
LR = 0.0001
NUM_CLASSES = len(train_dataset[0][1]) 

print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LR}")

Batch size: 32
Epochs: 5
Learning Rate: 0.0001


In [4]:
# ----- Tải và định nghĩa các DataLoader -----
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("DataLoader đã được tạo.")

DataLoader đã được tạo.


In [5]:
# ----- Khởi tạo Model -----
print("Khởi tạo mô hình ConvNeXt_Tiny...")
weights = ConvNeXt_Tiny_Weights.DEFAULT # Sử dụng trọng số pretrained mặc định
model = convnext_tiny(weights=weights)


model.classifier[2] = nn.Linear(model.classifier[2].in_features, NUM_CLASSES)
print(f"Kiến trúc mô hình đã thay đổi lớp phân loại cuối cùng thành {NUM_CLASSES} đầu ra.")

Khởi tạo mô hình ConvNeXt_Tiny...
Kiến trúc mô hình đã thay đổi lớp phân loại cuối cùng thành 6 đầu ra.


In [6]:
# ----- Cấu hình Device, Loss Function, Optimizer -----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Mô hình được chuyển đến thiết bị: {device}")

# Tính toán pos_weight cho BCEWithLogitsLoss để xử lý mất cân bằng lớp
positive_counts = torch.zeros(NUM_CLASSES)
negative_counts = torch.zeros(NUM_CLASSES)

print("Tính toán positive_counts và negative_counts cho pos_weight...")
for _, labels in tqdm(train_loader, desc="Calculating pos_weight"):
    labels = labels.float() # Đảm bảo labels là float (vì là BCEWithLogitsLoss)
    positive_counts += torch.sum(labels, dim=0)
    negative_counts += torch.sum(1 - labels, dim=0) # 1 - labels cho nhãn 0

# Tránh chia cho 0 nếu một lớp không có mẫu dương tính/âm tính nào
pos_weight = negative_counts / (positive_counts + 1e-5) 
pos_weight = pos_weight.to(device) 

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) # Dùng cho multi-label classification với trọng số
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Loss Function (BCEWithLogitsLoss với pos_weight) và Optimizer (Adam) đã được cấu hình.")
print(f"pos_weight: {pos_weight.cpu().numpy()}")

Mô hình được chuyển đến thiết bị: cpu
Tính toán positive_counts và negative_counts cho pos_weight...


Calculating pos_weight:   0%|          | 0/170 [00:00<?, ?it/s]

Calculating pos_weight: 100%|██████████| 170/170 [00:30<00:00,  5.51it/s]

Loss Function (BCEWithLogitsLoss với pos_weight) và Optimizer (Adam) đã được cấu hình.
pos_weight: [5.42800000e+08 1.59096575e+01 1.30292749e+00 7.86928082e+00
 9.42280502e+01 7.53229976e-01]


In [7]:
# ----- Hàm huấn luyện -----
def train():
    model.train() # Đặt mô hình ở chế độ huấn luyện
    print("Bắt đầu huấn luyện mô hình...")
    for epoch in range(EPOCHS):
        total_loss = 0
        # Sử dụng tqdm để hiển thị thanh tiến trình cho mỗi epoch
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            images = images.float().to(device)
            # labels đã là tensor float32 từ Custom Dataset của bạn, không cần stack lại.
            # Chỉ cần đảm bảo nó có đúng kích thước và chuyển lên device.
            labels = labels.float().to(device) 

            optimizer.zero_grad() # Xóa gradient của các tham số
            outputs = model(images) # Forward pass: lấy logits
            loss = criterion(outputs, labels) # Tính toán loss
            loss.backward() # Backward pass: tính gradient
            optimizer.step() # Cập nhật trọng số
            total_loss += loss.item() # Cộng dồn loss của batch

        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")
    print("Huấn luyện hoàn tất.")

In [8]:
# ----- Hàm đánh giá đơn giản (độ chính xác theo ngưỡng 0.5) -----
def evaluate(loader):
    model.eval() # Đặt mô hình ở chế độ đánh giá
    correct_predictions_per_label = torch.zeros(NUM_CLASSES)
    total_labels_per_label = torch.zeros(NUM_CLASSES)
    total_elements = 0

    print("Bắt đầu đánh giá độ chính xác đơn giản...")
    with torch.no_grad(): # Không cần tính gradient trong quá trình đánh giá
        for images, labels in tqdm(loader, desc="Evaluating simple accuracy"):
            images = images.float().to(device)
            labels = labels.float().to(device)

            outputs = model(images)
            predicted = (torch.sigmoid(outputs) > 0.5).float() # Dự đoán nhị phân với ngưỡng 0.5

            # Tính tổng số dự đoán đúng trên tất cả các nhãn trong batch
            overall_correct_batch = (predicted == labels).sum().item()
            overall_accuracy_batch = overall_correct_batch / torch.numel(labels)
            
            # Tính đúng trên từng nhãn (nếu cần, nhưng không dùng cho overall_accuracy)
            correct_predictions_per_label += ((predicted == labels) * labels).sum(dim=0).cpu()
            total_labels_per_label += labels.sum(dim=0).cpu()
            total_elements += torch.numel(labels) # Tổng số phần tử nhãn (batch_size * NUM_CLASSES)

    # Tính độ chính xác tổng thể (micro accuracy)
    overall_correct = (predicted == labels).sum().item() # Lấy của batch cuối cùng, cần phải tổng hợp toàn bộ
    
    total_correct_predictions = 0
    total_num_elements = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.float().to(device)
            labels = labels.float().to(device)
            outputs = model(images)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            
            total_correct_predictions += (predicted == labels).sum().item()
            total_num_elements += torch.numel(labels)

    if total_num_elements == 0:
        return 0.0
    return total_correct_predictions / total_num_elements

In [9]:
def evaluate_and_report(model, dataloader, label_list_topic):
    model.eval() # Đặt mô hình ở chế độ đánh giá
    device = "cuda" if torch.cuda.is_available() else "cpu" # Lấy lại device
    model.to(device) # Chuyển mô hình sang device nếu chưa có

    all_topic_preds = []
    all_topic_labels = []

    print("\nBắt đầu đánh giá chi tiết và thu thập dự đoán...")
    with torch.no_grad(): # Không cần tính gradient
        for images, labels in tqdm(dataloader, desc="Evaluating and collecting predictions"):
            images = images.float().to(device)
            labels = labels.float().to(device) # Labels cần là float cho BCEWithLogitsLoss

            outputs = model(images) # Forward pass
            
            # Lưu ground truth
            all_topic_labels.append(labels.detach().cpu())

            # Dự đoán topic: multi-label (sigmoid và ngưỡng 0.5)
            topic_preds = (torch.sigmoid(outputs) > 0.5).int().cpu() # Chuyển về int (0 hoặc 1)
            all_topic_preds.append(topic_preds)

    # Nối lại toàn bộ các tensor đã thu thập
    y_topic_true = torch.cat(all_topic_labels).numpy()
    y_topic_pred = torch.cat(all_topic_preds).numpy()

    print("\n📊 Topic Classification Report:")
    # Chú ý: zero_division=0 để tránh cảnh báo khi precision/recall là 0
    print(classification_report(y_topic_true, y_topic_pred, target_names=label_list_topic, zero_division=0))

    print("\n🧩 Confusion Matrix for Topics:")
    topic_cm = multilabel_confusion_matrix(y_topic_true, y_topic_pred)
    # multilabel_confusion_matrix trả về một ma trận cho mỗi nhãn
    for i, cm in enumerate(topic_cm):
        print(f"Topic: {label_list_topic[i]}")
        print(cm)
        print("")
    print("Đánh giá chi tiết hoàn tất.")

In [10]:
# ----- Huấn luyện và đánh giá -----
print("--- Bắt đầu quá trình huấn luyện ---")
train()
print("--- Kết thúc quá trình huấn luyện ---")

# Gọi hàm đánh giá chi tiết trên tập dev
print("\n--- Đánh giá mô hình trên tập Development ---")
evaluate_and_report(model, dev_loader, label_columns) # Dùng label_columns từ preprocessing

# Lưu mô hình đã huấn luyện
MODEL_SAVE_PATH = "model.pth"
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"\nMô hình đã được lưu tại: {MODEL_SAVE_PATH}")

# Đánh giá cuối cùng trên tập test
print("\n--- Đánh giá mô hình trên tập Test ---")
acc = evaluate(test_loader)
print(f"Độ chính xác trên tập test (micro accuracy): {acc*100:.2f}%")

--- Bắt đầu quá trình huấn luyện ---
Bắt đầu huấn luyện mô hình...


Epoch 1/5: 100%|██████████| 170/170 [06:35<00:00,  2.33s/it]


Epoch 1/5, Loss: 78.5085


Epoch 2/5: 100%|██████████| 170/170 [05:40<00:00,  2.00s/it]


Epoch 2/5, Loss: 36.8988


...

Epoch 5/5, Loss: 8.4617
Huấn luyện hoàn tất.
--- Kết thúc quá trình huấn luyện ---

--- Đánh giá mô hình trên tập Development ---

Bắt đầu đánh giá chi tiết và thu thập dự đoán...


Evaluating and collecting predictions: 100%|██████████| 56/56 [01:01<00:00,  1.09s/it]



📊 Topic Classification Report:
              precision    recall  f1-score   support

    Location       0.00      0.00      0.00         0
        Food       0.91      0.90      0.91       101
        Room       0.91      0.93      0.92       795
  Facilities       0.86      0.85      0.85       222
     Service       0.43      0.22      0.29        27
 Public_area       0.94      0.91      0.92      1020

   micro avg       0.91      0.90      0.91      2165
   macro avg       0.67      0.64      0.65      2165
weighted avg       0.91      0.90      0.91      2165
 samples avg       0.93      0.93      0.92      2165


🧩 Confusion Matrix for Topics:
Topic: Location
[[1789    0]
 [   0    0]]

Topic: Food
[[1679    9]
 [  10   91]]

Topic: Room
[[920  74]
 [ 52 743]]

Topic: Facilities
[[1535   32]
 [  33  189]]

Topic: Service
[[1754    8]
 [  21    6]]

Topic: Public_area
[[705  64]
 [ 91 929]]

Đánh giá chi tiết hoàn tất.

Mô hình đã được lưu tại: model.pth

--- Đánh giá mô hình t

Evaluating simple accuracy: 100%|██████████| 58/58 [01:05<00:00,  1.13s/it]


Độ chính xác trên tập test (micro accuracy): 96.51%
